# Track A — Random Forest (Karan)

Hourly heat demand per building type (`profile_type`). Data: `data/modeling/hourly_heat.parquet`
(17 profile types x 8,760 hours of 2018). Train Jan-Sep, predict Oct-Dec, score against
the baseline (0.392 mean CV(RMSE)) with `scripts/score.py`.

Features: weather + calendar + `profile_type` only - no past demand. Target: `heat_intensity`,
converted back to kWh with `* scale_value` before scoring.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("couldn't find repo root (pyproject.toml)")


ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

from heatreuse import io, config, timeutils, units, metrics
from heatreuse.forecasting.splits import month_folds

pd.set_option("display.width", 120)

# reset_index so positions line up exactly with month_folds' own internal
# ts.reset_index(drop=True) - every array below (X_all, y_all, ...) stays in this order.
hourly = pd.read_parquet(ROOT / "data" / "modeling" / "hourly_heat.parquet").reset_index(drop=True)
print(hourly.shape)
print(hourly["data_split"].value_counts())
hourly.head()

## 2. Quick EDA — profile types and the heat-vs-temperature shape

`low_model_count` flags the types built from few simulated buildings (noisier signal) —
worth knowing before the diagnostics section later.

In [ ]:
profile_info = (hourly.groupby("profile_type", observed=True)
                .agg(models_used=("models_used", "first"),
                     low_model_count=("low_model_count", "first"),
                     scale_basis=("scale_basis", "first"))
                .sort_values("models_used"))
profile_info

In [ ]:
types = sorted(hourly["profile_type"].unique())
fig, axes = plt.subplots(5, 4, figsize=(16, 16), sharex=True)
for ax, ptype in zip(axes.flat, types):
    g = hourly[hourly["profile_type"] == ptype]
    ax.scatter(g["temp_c"], g["heat_intensity"], s=2, alpha=0.2)
    ax.set_title(ptype, fontsize=8)
for ax in axes.flat[len(types):]:
    ax.axis("off")
fig.suptitle("heat_intensity vs temp_c by profile_type (sanity check: should fall as temp rises)")
fig.tight_layout()
plt.show()

## 3. Features and target

`profile_type` one-hot encoded, not ordinal: `RandomForestRegressor` has no native
categorical support, and ordinal encoding would impose a fake order on 17 unrelated
building types. One-hot is fit on the whole table once up front — the 17 categories are
a fixed, known set, so there is no leakage from doing this before any CV split.

`month` is built separately as a toggle, decided empirically in section 4.

In [ ]:
BASE_FEATURES = ["temp_c", "rel_humidity_pct", "wind_speed_ms", "ghi_wm2", "hdh_18c",
                 "temp_c_roll24", "temp_c_lag1", "temp_c_lag3",
                 "hour_of_day", "day_of_week", "is_weekend", "is_holiday"]
TARGET = "heat_intensity"


def make_design_matrix(df: pd.DataFrame, include_month: bool) -> pd.DataFrame:
    feats = BASE_FEATURES + (["month"] if include_month else [])
    X = df[feats].copy()
    X["is_weekend"] = X["is_weekend"].astype(int)
    X["is_holiday"] = X["is_holiday"].astype(int)
    X["profile_type"] = df["profile_type"]
    return pd.get_dummies(X, columns=["profile_type"])


X_all_with_month = make_design_matrix(hourly, include_month=True)
X_all_no_month = make_design_matrix(hourly, include_month=False)
y_all = hourly[TARGET].to_numpy()

print(X_all_no_month.shape, X_all_with_month.shape)
X_all_no_month.head()

## 4. Month feature: with vs without. OOB vs honest CV.

This section has three parts:
- **4a** — the original leave-one-month-out CV comparison, kept for the Jan/Feb breakdown,
  with pooled (whole-season) scoring added to fix a real problem in the plain mean.
- **4b** — the actual decision: a Mar-Sep-train / Jan-Feb-validate check that mimics real
  extrapolation, since `month_folds` can't.
- **4c** — OOB vs honest pooled CV, once, on the feature set `4b` settles on.

**Note on `n_estimators`:** a single 300-tree fit benchmarked at ~145s on this machine
even with `n_jobs=-1` across 10 cores (a 50-tree fit only dropped to ~27s, so the ceiling
is per-tree cost, not missing parallelism) — running the full search at 300 trees
throughout would take hours. Every CV/search fit in this section and sections 4b-5 uses
`n_estimators=50`; only the one final refit in section 6 uses the full
`n_estimators=300`, so the actual submission isn't affected.

In [ ]:
def mean_cvrmse_by_type(y, pred, profile_type):
    """Same recipe as scripts/score.py's score_hourly: cvrmse per profile_type, then mean."""
    df = pd.DataFrame({"profile_type": np.asarray(profile_type), "y": y, "pred": pred})
    per_type = {t: metrics.cvrmse(g["y"].to_numpy(), g["pred"].to_numpy())
                for t, g in df.groupby("profile_type", observed=True)}
    s = pd.Series(per_type)
    return s.mean(), s


def type_means(y_raw, profile_type, idx):
    """Per-profile_type mean of y_raw over rows at positions ``idx`` only (e.g. a fold's
    training rows) - used to normalize the target so every type weighs equally (section 4d)."""
    s = pd.Series(np.asarray(y_raw)[idx], index=np.asarray(profile_type)[idx])
    return s.groupby(level=0, observed=True).mean()


def oof_cv(X, y, ts, profile_type, rf_kwargs, normalize_by_type=False):
    """One RandomForestRegressor per month_folds fold (Jan-Sep, 168h gap).

    Returns:
      per_fold       - DataFrame[month, mean_cvrmse]: per-fold cvrmse, display only (it's
                       unstable in summer - see the note below) - keeps Jan/Feb visible.
      pooled         - float: cvrmse per profile_type computed ONCE over every concatenated
                       out-of-fold prediction (all 9 months together), then mean across
                       types - the same structure scripts/score.py uses for Oct-Dec. This
                       is the number to actually compare models on.
      pooled_by_type - the per-type Series backing ``pooled``.

    If normalize_by_type, the model trains on y / (that fold's training-only per-type
    mean) so every type contributes about equally to the squared-error loss; predictions
    are scaled back by the same training-only mean before any score is computed, so
    cvrmse is always reported in the original heat_intensity units.
    """
    oof_pred = np.full(len(y), np.nan)
    fold_rows = []
    profile_type = pd.Series(np.asarray(profile_type))
    for month, trn_idx, val_idx in month_folds(ts):
        if normalize_by_type:
            tmeans = type_means(y, profile_type, trn_idx)
            scale_trn = profile_type.iloc[trn_idx].map(tmeans).to_numpy()
            scale_val = profile_type.iloc[val_idx].map(tmeans).to_numpy()
            rf = RandomForestRegressor(random_state=0, n_jobs=-1, **rf_kwargs)
            rf.fit(X.iloc[trn_idx], y[trn_idx] / scale_trn)
            pred = rf.predict(X.iloc[val_idx]) * scale_val
        else:
            rf = RandomForestRegressor(random_state=0, n_jobs=-1, **rf_kwargs)
            rf.fit(X.iloc[trn_idx], y[trn_idx])
            pred = rf.predict(X.iloc[val_idx])
        oof_pred[val_idx] = pred
        mean_cv, _ = mean_cvrmse_by_type(y[val_idx], pred, profile_type.iloc[val_idx])
        fold_rows.append({"month": month, "mean_cvrmse": mean_cv})
    has_pred = ~np.isnan(oof_pred)
    pooled, pooled_by_type = mean_cvrmse_by_type(y[has_pred], oof_pred[has_pred], profile_type[has_pred])
    return pd.DataFrame(fold_rows), pooled, pooled_by_type

In [ ]:
# 4a. 9 folds x 2 feature sets = 18 fits. n_estimators=50 for this search pass (see
# the note on n_estimators in the original section 4 markdown above).
SEARCH_N_ESTIMATORS = 50
fold_with, pooled_with, _ = oof_cv(X_all_with_month, y_all, hourly["timestamp"], hourly["profile_type"],
                                   rf_kwargs=dict(n_estimators=SEARCH_N_ESTIMATORS))
fold_no, pooled_no, _ = oof_cv(X_all_no_month, y_all, hourly["timestamp"], hourly["profile_type"],
                               rf_kwargs=dict(n_estimators=SEARCH_N_ESTIMATORS))

print(f"pooled (whole Jan-Sep, one cvrmse per type then mean) - with month: {pooled_with:.4f}")
print(f"pooled (whole Jan-Sep, one cvrmse per type then mean) - no month:   {pooled_no:.4f}")
pd.DataFrame({"with_month": fold_with.set_index("month")["mean_cvrmse"],
             "no_month": fold_no.set_index("month")["mean_cvrmse"]})

**Why the per-fold mean would be misleading, and why pooling fixes it:**
`cvrmse = RMSE / mean(actual)`. In Jun/Jul/Aug several building types have `heat_intensity`
near zero, so a single month's denominator is tiny and that month's `cvrmse` explodes (the
per-fold table above shows July/August in the hundreds) — a numerical artifact, not a real
signal. Pooling every fold's out-of-fold prediction across the full Jan-Sep span before
computing `cvrmse` (the `pooled` numbers above) fixes this exactly the way
`scripts/score.py` avoids it for Oct-Dec: one `cvrmse` per `profile_type` over the *whole*
season, not an average of single-month snapshots, so a type's few near-zero summer hours
don't dominate its own full-season denominator.

**This still isn't a real test of `month`, though.** `month_folds` only ever holds out one
month from the *interior* of Jan-Sep — the model always has, say, both March and May in
training when validating April, so it's interpolating between neighbouring months it
already knows, never extrapolating past the edge of its training range the way it will
have to for the real Oct-Dec test. The actual decision is next.

### 4b. The real extrapolation check: train Mar–Sep, validate Jan–Feb

This is what actually decides whether `month` stays in the final feature set. Train on
Mar–Sep only and validate on Jan–Feb — held out at the *edge* of the training range, the
same way Oct–Dec sits past the edge of the real Jan–Sep training set — rather than
leave-one-month-out, which only ever interpolates. Both halves come from
`data_split == "train"`; the real Oct–Dec test set is never touched by this decision.

In [ ]:
train_mask = (hourly["data_split"] == "train").to_numpy()

extrap_train_mask = train_mask & hourly["month"].between(3, 9).to_numpy()
extrap_val_mask = train_mask & hourly["month"].isin([1, 2]).to_numpy()
print(f"extrapolation train rows (Mar-Sep): {extrap_train_mask.sum()}, "
     f"validate rows (Jan-Feb): {extrap_val_mask.sum()}")

extrap_rows = []
for label, X in [("with_month", X_all_with_month), ("without_month", X_all_no_month)]:
    rf = RandomForestRegressor(random_state=0, n_jobs=-1, n_estimators=SEARCH_N_ESTIMATORS)
    rf.fit(X.loc[extrap_train_mask], y_all[extrap_train_mask])
    pred = rf.predict(X.loc[extrap_val_mask])
    mean_cv, _ = mean_cvrmse_by_type(y_all[extrap_val_mask], pred, hourly.loc[extrap_val_mask, "profile_type"])
    extrap_rows.append({"features": label, "mean_cvrmse": mean_cv})

extrap_df = pd.DataFrame(extrap_rows).set_index("features")
USE_MONTH = bool(extrap_df.loc["with_month", "mean_cvrmse"] < extrap_df.loc["without_month", "mean_cvrmse"])
X_all = X_all_with_month if USE_MONTH else X_all_no_month
print("Using month feature (decided by the Mar-Sep -> Jan-Feb extrapolation check):", USE_MONTH)
extrap_df

### 4c. OOB vs honest (pooled) CV, once

OOB only holds out individual bootstrapped rows. Because consecutive hours are highly
autocorrelated, a left-out row's near-neighbors are almost always still in-bag for some
tree, so OOB understates error on a genuinely unseen block of time — which is exactly
what the real Oct-Dec test is. Compared here against the pooled out-of-fold CV score
(section 4a) on the feature set 4b chose — never OOB, and never an arbitrary
month_folds mean/median.

In [ ]:
# Same SEARCH_N_ESTIMATORS as the CV comparison above, so this is an apples-to-apples
# OOB-vs-CV comparison at matched tree count (not a comparison against the final model).
rf_oob = RandomForestRegressor(n_estimators=SEARCH_N_ESTIMATORS, oob_score=True, n_jobs=-1, random_state=0)
rf_oob.fit(X_all.loc[train_mask], y_all[train_mask])

oob_mean_cvrmse, _ = mean_cvrmse_by_type(
    y_all[train_mask], rf_oob.oob_prediction_, hourly.loc[train_mask, "profile_type"])

# Reuse whichever pooled score (section 4a) matches the chosen feature set - same fold
# structure, same n_estimators, no need to refit 9 more models.
pooled_cv = pooled_with if USE_MONTH else pooled_no
print(f"OOB mean CV(RMSE):      {oob_mean_cvrmse:.4f}")
print(f"Honest pooled CV(RMSE): {pooled_cv:.4f}")

### 4d. Equal-weighting by type (normalize target by each type's training mean)

The scorer weights every `profile_type` equally (mean `cvrmse` across types), but raw
`heat_intensity` doesn't — residential types run far higher intensity than commercial
ones (see the permutation-importance finding below), so they'd dominate the
squared-error loss during training even though the scorer doesn't care more about them.
Training on `heat_intensity / (that type's OWN training-period mean)` instead makes
every type contribute on a comparable scale; predictions are scaled back by the same
training-only mean before any score (or the final submission) is computed, so this never
changes the units being evaluated.

In [ ]:
# Reuse the pooled score from section 4a for the chosen feature set (identical fold
# structure, rf_kwargs and random_state - no need to refit 9 more raw-target models).
pooled_plain = pooled_with if USE_MONTH else pooled_no
_, pooled_norm, _ = oof_cv(X_all, y_all, hourly["timestamp"], hourly["profile_type"],
                          rf_kwargs=dict(n_estimators=SEARCH_N_ESTIMATORS), normalize_by_type=True)

print(f"pooled CV(RMSE), raw heat_intensity:      {pooled_plain:.4f}")
print(f"pooled CV(RMSE), normalized by type mean: {pooled_norm:.4f}")

NORMALIZE_BY_TYPE = bool(pooled_norm < pooled_plain)
print("Using per-type normalization:", NORMALIZE_BY_TYPE)

## 5. Tuning grid (`n_estimators=50` for search — see the note in section 4; the final
## submission in section 6 refits at the full `n_estimators=300`)

`max_features x min_samples_leaf`, 9 combinations, ranked by the pooled out-of-fold
CV(RMSE) (never OOB, never a single-month mean/median) over the 9 `month_folds` folds,
using whichever feature set (4b) and target scheme (4d) were chosen above.
`max_features=1.0` with row bootstrapping is plain bagging; `max_features<1.0` adds the
random-subspace trick that turns bagging into a proper Random Forest — this grid is a
direct, presentable answer to how much that buys over bagging alone.

If this cell is still too slow on your machine, add `max_samples=0.5` to `kwargs` below
and re-run — not included by default, only as a fallback.

In [ ]:
import itertools
import time

GRID = list(itertools.product([0.33, 0.5, 1.0], [1, 5, 20]))  # (max_features, min_samples_leaf)
grid_results = []
t0 = time.time()
for max_features, min_samples_leaf in GRID:
    kwargs = dict(n_estimators=SEARCH_N_ESTIMATORS, max_features=max_features,
                 min_samples_leaf=min_samples_leaf, max_depth=None)
    fold_tbl, pooled, _ = oof_cv(X_all, y_all, hourly["timestamp"], hourly["profile_type"],
                                 rf_kwargs=kwargs, normalize_by_type=NORMALIZE_BY_TYPE)
    grid_results.append({"max_features": max_features, "min_samples_leaf": min_samples_leaf,
                         "pooled_cvrmse": pooled,
                         "jan_cvrmse": fold_tbl.loc[fold_tbl["month"] == 1, "mean_cvrmse"].iloc[0],
                         "feb_cvrmse": fold_tbl.loc[fold_tbl["month"] == 2, "mean_cvrmse"].iloc[0]})
    print(grid_results[-1], f"  ({time.time() - t0:.0f}s elapsed)")

grid_df = pd.DataFrame(grid_results).sort_values("pooled_cvrmse").reset_index(drop=True)
grid_df

## 6. Final fit on all Jan–Sep, predict Oct–Dec once

Refit with the winning grid settings on every training row, predict the test rows
exactly once, convert back to kWh, and clip negatives to 0 — mirroring the same
`np.clip(..., 0, None)` pattern `scripts/baselines.py` already uses.

In [ ]:
best = grid_df.iloc[0]
BEST_KWARGS = dict(n_estimators=300, max_features=best["max_features"],
                   min_samples_leaf=int(best["min_samples_leaf"]), max_depth=None)
print("Best hyperparameters:", BEST_KWARGS, " USE_MONTH =", USE_MONTH,
     " NORMALIZE_BY_TYPE =", NORMALIZE_BY_TYPE)

test_mask = (hourly["data_split"] == "test").to_numpy()
final_rf = RandomForestRegressor(random_state=0, n_jobs=-1, **BEST_KWARGS)

if NORMALIZE_BY_TYPE:
    final_type_means = type_means(y_all, hourly["profile_type"], np.flatnonzero(train_mask))
    scale_trn = hourly.loc[train_mask, "profile_type"].map(final_type_means).to_numpy()
    scale_test = hourly.loc[test_mask, "profile_type"].map(final_type_means).to_numpy()
    final_rf.fit(X_all.loc[train_mask], y_all[train_mask] / scale_trn)
    pred_intensity = final_rf.predict(X_all.loc[test_mask]) * scale_test
else:
    final_rf.fit(X_all.loc[train_mask], y_all[train_mask])
    pred_intensity = final_rf.predict(X_all.loc[test_mask])

predicted_kwh = np.clip(pred_intensity * hourly.loc[test_mask, "scale_value"].to_numpy(), 0, None)

submission = pd.DataFrame({
    "timestamp": hourly.loc[test_mask, "timestamp"].to_numpy(),
    "profile_type": hourly.loc[test_mask, "profile_type"].to_numpy(),
    "predicted_kwh": predicted_kwh,
})
submission.shape

## 7. Pre-submission checks

Replicates `scripts/score.py`'s own merge check before running it for real, so a bad
submission file is caught here with a clear message rather than via its `SystemExit`.

In [ ]:
assert len(submission) == 37536, len(submission)
assert submission["predicted_kwh"].isna().sum() == 0
assert (submission["predicted_kwh"] < 0).sum() == 0
assert not submission.duplicated(["timestamp", "profile_type"]).any()
assert set(submission["profile_type"]) == set(hourly["profile_type"].unique())

truth_test = hourly.loc[test_mask, ["timestamp", "profile_type"]]
check = truth_test.merge(submission, on=["timestamp", "profile_type"], how="outer", indicator=True)
assert (check["_merge"] == "both").all(), check["_merge"].value_counts()

print("all checks passed:", len(submission), "rows")

## 8. Write submission and score it

Scores via the real `scripts/score.py` subprocess rather than reimplementing its logic —
it's the one shared path everyone's score goes through, and it's what appends to
`submissions/leaderboard.csv`. The printed mean CV(RMSE) here, not any CV number above,
is what's comparable to the 0.392 baseline.

In [ ]:
sub_path = ROOT / "submissions" / "karan_hourly.csv"
submission.to_csv(sub_path, index=False)
print("wrote", sub_path)

In [ ]:
import subprocess

result = subprocess.run(["python", "scripts/score.py", "hourly", "submissions/karan_hourly.csv"],
                        cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scoring failed")

## 9. Diagnostics

Per-type `cvrmse`/`nmbe`/`wape` is already printed by `score.py` above — tie the worst
types back to `low_model_count` from section 2.

Permutation importance (not impurity importance) is computed on the **January
validation fold** from `month_folds`, not the final test set, using
`scoring="neg_root_mean_squared_error"` so results are in real RMSE-increase units.
Impurity-based importance is known to inflate high-cardinality / many-dummy-column
features, which matters here specifically because `profile_type` is one-hot encoded
into 17 columns — so the 17 `profile_type` columns are also permuted **together as one
group** (shuffling which row gets which type's whole one-hot block, not each column
independently), which is the fair way to ask "how much does knowing the building type
matter" when that one piece of information is spread across 17 correlated columns.

In [ ]:
class RealUnitsPredictor:
    """Wraps a model trained on a normalized target so .predict() returns real
    heat_intensity units, using each row's OWN true per-type scale factor (computed
    once from the unpermuted validation data and held fixed) - so permutation_importance
    shuffling a feature column only affects the model's prediction, never which scale
    factor gets applied back. Forwards everything else to the wrapped model so sklearn's
    internal checks (fitted attrs, etc.) still resolve."""
    def __init__(self, model, scale):
        self.model = model
        self.scale = np.asarray(scale)

    def predict(self, X):
        return self.model.predict(X) * self.scale

    def __getattr__(self, name):
        return getattr(self.model, name)


def grouped_permutation_importance(estimator, X, y, group_cols, n_repeats=5, random_state=0):
    """Permute a GROUP of columns together (as one feature), for correlated dummy
    columns like one-hot profile_type. Reports the RMSE increase (same sign/scale as
    permutation_importance with scoring='neg_root_mean_squared_error': positive = a
    helpful feature), so it is directly comparable to the per-feature values."""
    rng = np.random.default_rng(random_state)
    baseline_rmse = np.sqrt(np.mean((y - estimator.predict(X)) ** 2))
    drops = []
    for _ in range(n_repeats):
        Xp = X.copy()
        perm = rng.permutation(len(X))
        Xp[group_cols] = X[group_cols].to_numpy()[perm]
        rmse = np.sqrt(np.mean((y - estimator.predict(Xp)) ** 2))
        drops.append(rmse - baseline_rmse)
    return float(np.mean(drops)), float(np.std(drops))


jan_month, jan_trn_idx, jan_val_idx = next(mf for mf in month_folds(hourly["timestamp"]) if mf[0] == 1)
profile_type_s = hourly["profile_type"]

rf_jan = RandomForestRegressor(random_state=0, n_jobs=-1, **BEST_KWARGS)
if NORMALIZE_BY_TYPE:
    jan_type_means = type_means(y_all, profile_type_s, jan_trn_idx)
    jan_scale_trn = profile_type_s.iloc[jan_trn_idx].map(jan_type_means).to_numpy()
    rf_jan.fit(X_all.iloc[jan_trn_idx], y_all[jan_trn_idx] / jan_scale_trn)
    jan_scale_val = profile_type_s.iloc[jan_val_idx].map(jan_type_means).to_numpy()
    rf_jan_for_importance = RealUnitsPredictor(rf_jan, jan_scale_val)
else:
    rf_jan.fit(X_all.iloc[jan_trn_idx], y_all[jan_trn_idx])
    rf_jan_for_importance = rf_jan

ptype_cols = [c for c in X_all.columns if c.startswith("profile_type_")]
weather_calendar_cols = [c for c in X_all.columns if c not in ptype_cols]

perm = permutation_importance(rf_jan_for_importance, X_all.iloc[jan_val_idx], y_all[jan_val_idx],
                              n_repeats=5, random_state=0, n_jobs=-1,
                              scoring="neg_root_mean_squared_error")
individual_importances = pd.Series(perm.importances_mean, index=X_all.columns)

grouped_mean, grouped_std = grouped_permutation_importance(
    rf_jan_for_importance, X_all.iloc[jan_val_idx], y_all[jan_val_idx], ptype_cols,
    n_repeats=5, random_state=0)

combined = individual_importances[weather_calendar_cols].copy()
combined["profile_type (grouped)"] = grouped_mean
combined = combined.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
combined.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_title("Permutation importance (RMSE increase), January validation fold\n"
             "profile_type's 17 one-hot columns permuted together as one group")
fig.tight_layout()
plt.show()

In [ ]:
truth_kwh = hourly.loc[test_mask, ["timestamp", "profile_type", "heat_total_kwh"]]
scored = truth_kwh.merge(submission, on=["timestamp", "profile_type"])
per_type_cvrmse = pd.Series({t: metrics.cvrmse(g["heat_total_kwh"], g["predicted_kwh"])
                            for t, g in scored.groupby("profile_type", observed=True)}).sort_values()

worst_type = per_type_cvrmse.index[-1]
typical_type = per_type_cvrmse.index[len(per_type_cvrmse) // 2]
print("worst:  ", worst_type, per_type_cvrmse.iloc[-1])
print("typical:", typical_type, per_type_cvrmse.loc[typical_type])

for t in [worst_type, typical_type]:
    g = scored[(scored["profile_type"] == t) & (scored["timestamp"].dt.month == 12)].sort_values("timestamp")
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(g["timestamp"], g["heat_total_kwh"], label="actual")
    ax.plot(g["timestamp"], g["predicted_kwh"], label="predicted", alpha=0.8)
    ax.set_title(f"{t} — December actual vs predicted")
    ax.legend()
    fig.tight_layout()
    plt.show()

## 10. Write-up

> Trained a single global `RandomForestRegressor` (scikit-learn, `n_estimators=300`,
> `max_features=0.5`, `min_samples_leaf=1`) on raw `heat_intensity` across all 17
> building types, using weather + calendar + one-hot `profile_type`, with `month`
> included (no past demand, which this dataset doesn't include at all). Whether `month`
> helps turned out to be close to a toss-up, not the clear win an earlier (flawed)
> leave-one-month-out check suggested: a validation that actually mimics extrapolation
> (train Mar–Sep, validate on the held-out edge months Jan–Feb, never Oct–Dec) scored
> 0.436 with `month` vs. 0.437 without — month_folds can only interpolate between
> neighbouring months it already knows, so it isn't a fair test of what happens at the
> real train/test boundary. I also tried training on `heat_intensity` normalized by each
> type's own training mean, so all 17 types weigh equally in the loss the way the scorer
> weighs them in the score — it didn't help (pooled CV(RMSE) 0.527 vs. 0.518 raw), so the
> final model keeps the raw target. Every CV number above is a *pooled* out-of-fold
> score — one `cvrmse` per type over the whole concatenated Jan–Sep span, the same
> structure `scripts/score.py` uses for Oct–Dec — because a plain mean/median of
> per-month scores is dominated by a real artifact: several building types have near-zero
> `heat_intensity` in Jun–Aug, so `cvrmse`'s denominator collapses and single-month scores
> explode into the hundreds. On that honest basis, OOB (0.177) was still about 2.9x more
> optimistic than true CV (0.518), confirming it shouldn't be used for model selection —
> neighbouring hours are near-duplicates, so OOB barely tests extrapolation to an unseen
> block of time. On the real Oct–Dec test set, `scripts/score.py` gave a **mean CV(RMSE)
> of 0.2667** against the 0.392 baseline — a **beat by 0.125 (32% relative
> improvement)**. Grouped permutation importance (the 17 one-hot `profile_type` columns
> permuted together as one feature, not individually) shows building type dominates by
> far the most — its RMSE-increase is roughly 10x the next-largest feature
> (`temp_c_lag1`) — while `month`'s importance is close to zero, consistent with the
> extrapolation check's near-tie. The model is worst on **warehouse** (CV(RMSE) = 0.522)
> — notably *not* a `low_model_count` type (114 simulated buildings backing it), so the
> error isn't from a thin simulated sample; warehouses likely just have more
> operationally irregular, less temperature-driven heating schedules that weather +
> calendar alone don't capture well. The best-performing type was
> `fullservicerestaurant` (CV(RMSE) = 0.092); a representative "typical" type was
> `outpatient` (CV(RMSE) = 0.280, see the December actual-vs-predicted plots above for
> both).